# 02 - Project Selection

Mục tiêu:
1. Thống kê các project trong 16 Jira repositories.
2. Đếm số issue của từng project.
3. Xác định số issue đã resolve và chưa resolve.
4. Kiểm tra mức độ đầy đủ của dữ liệu.
5. Lựa chọn tập project dùng cho thực nghiệm cross-project.

In [1]:
from pymongo import MongoClient
import pandas as pd
import numpy as np

client = MongoClient("mongodb://localhost:27017/")

print(client.admin.command("ping"))

{'ok': 1.0}


In [6]:
DB_NAME = "JiraReposAnon"

db = client[DB_NAME]

In [7]:
collections = sorted(db.list_collection_names())

collections

['Apache',
 'Hyperledger',
 'IntelDAOS',
 'JFrog',
 'Jira',
 'JiraEcosystem',
 'MariaDB',
 'Mindville',
 'Mojang',
 'MongoDB',
 'Qt',
 'RedHat',
 'Sakai',
 'SecondLife',
 'Sonatype',
 'Spring']

In [8]:
PROJECT_FIELD = "fields.project.key"
CREATED_FIELD = "fields.created"
RESOLUTION_FIELD = "fields.resolutiondate"

In [9]:
collection = db["Apache"]

In [10]:
pipeline = [
    {
        "$match": {
            PROJECT_FIELD: {"$ne": None}
        }
    },
    {
        "$group": {
            "_id": f"${PROJECT_FIELD}",
            "issues": {"$sum": 1}
        }
    },
    {
        "$sort": {
            "issues": -1
        }
    }
]

In [11]:
apache_projects = list(
    collection.aggregate(
        pipeline,
        allowDiskUse=True
    )
)

apache_projects[:10]

[{'_id': 'SPARK', 'issues': 37443},
 {'_id': 'FLEX', 'issues': 35390},
 {'_id': 'HBASE', 'issues': 26421},
 {'_id': 'HIVE', 'issues': 25731},
 {'_id': 'FLINK', 'issues': 25492},
 {'_id': 'AMBARI', 'issues': 25384},
 {'_id': 'CAMEL', 'issues': 17391},
 {'_id': 'CASSANDRA', 'issues': 17115},
 {'_id': 'IGNITE', 'issues': 16194},
 {'_id': 'HADOOP', 'issues': 15797}]

In [12]:
pipeline = [
    {
        "$match": {
            PROJECT_FIELD: {"$ne": None}
        }
    },
    {
        "$group": {
            "_id": f"${PROJECT_FIELD}",

            "issues": {
                "$sum": 1
            },

            "resolved": {
                "$sum": {
                    "$cond": [
                        {"$ne": [f"${RESOLUTION_FIELD}", None]},
                        1,
                        0
                    ]
                }
            },

            "unresolved": {
                "$sum": {
                    "$cond": [
                        {"$eq": [f"${RESOLUTION_FIELD}", None]},
                        1,
                        0
                    ]
                }
            }
        }
    },
    {
        "$sort": {
            "issues": -1
        }
    }
]

In [13]:
apache_stats = list(
    db["Apache"].aggregate(
        pipeline,
        allowDiskUse=True
    )
)

pd.DataFrame(apache_stats).head(20)

,_id,issues,resolved,unresolved
0,SPARK,37443,35034,2409
1,FLEX,35390,31564,3826
2,HBASE,26421,22774,3647
3,HIVE,25731,18516,7215
4,FLINK,25492,21398,4094
5,AMBARI,25384,23305,2079
6,CAMEL,17391,16716,675
7,CASSANDRA,17115,14816,2299
8,IGNITE,16194,11949,4245
9,HADOOP,15797,13217,2580


In [14]:
all_project_stats = []

for repo in collections:

    print("Processing:", repo)

    pipeline = [
        {
            "$match": {
                PROJECT_FIELD: {"$ne": None}
            }
        },
        {
            "$group": {
                "_id": f"${PROJECT_FIELD}",

                "issues": {
                    "$sum": 1
                },

                "resolved": {
                    "$sum": {
                        "$cond": [
                            {"$ne": [f"${RESOLUTION_FIELD}", None]},
                            1,
                            0
                        ]
                    }
                },

                "unresolved": {
                    "$sum": {
                        "$cond": [
                            {"$eq": [f"${RESOLUTION_FIELD}", None]},
                            1,
                            0
                        ]
                    }
                }
            }
        }
    ]

    results = db[repo].aggregate(
        pipeline,
        allowDiskUse=True
    )

    for row in results:

        all_project_stats.append({
            "repository": repo,
            "project": row["_id"],
            "issues": row["issues"],
            "resolved": row["resolved"],
            "unresolved": row["unresolved"]
        })

Processing: Apache
Processing: Hyperledger
Processing: IntelDAOS
Processing: JFrog
Processing: Jira
Processing: JiraEcosystem
Processing: MariaDB
Processing: Mindville
Processing: Mojang
Processing: MongoDB
Processing: Qt
Processing: RedHat
Processing: Sakai
Processing: SecondLife
Processing: Sonatype
Processing: Spring


In [15]:
projects_df = pd.DataFrame(all_project_stats)

projects_df.head()

,repository,project,issues,resolved,unresolved
0,Apache,TOREE,528,411,117
1,Apache,XALANJ,954,697,257
2,Apache,VELOCITYSB,9,9,0
3,Apache,JSIEVE,115,100,15
4,Apache,ACE,539,492,47


In [16]:
projects_df["resolved_rate"] = (
    projects_df["resolved"]
    / projects_df["issues"]
)

projects_df["censoring_rate"] = (
    projects_df["unresolved"]
    / projects_df["issues"]
)

In [17]:
projects_df.sort_values(
    "issues",
    ascending=False
).head(30)

,repository,project,issues,resolved,unresolved,resolved_rate,censoring_rate
844,Mojang,MC,213845,205547,8298,0.961196,0.038804
841,Mojang,MCPE,144778,140317,4461,0.969187,0.030813
884,Qt,QTBUG,97172,77520,19652,0.797761,0.202239
1192,Sonatype,OSSRH,74960,72193,2767,0.963087,0.036913
867,MongoDB,SERVER,58928,52972,5956,0.898928,0.101072
690,Jira,JRASERVER,47225,38678,8547,0.819015,0.180985
710,Jira,CONFSERVER,43910,37290,6620,0.849237,0.150763
1180,Sakai,SAK,43351,39086,4265,0.901617,0.098383
293,Apache,SPARK,37443,35034,2409,0.935662,0.064338
313,Apache,FLEX,35390,31564,3826,0.891890,0.108110


In [18]:
print(
    "Repositories:",
    projects_df["repository"].nunique()
)

print(
    "Projects:",
    len(projects_df)
)

print(
    "Total issues:",
    projects_df["issues"].sum()
)

Repositories: 16
Projects: 1276
Total issues: 2686282


In [19]:
from pathlib import Path

output_dir = Path("../results/tables")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

projects_df.to_csv(
    output_dir / "project_inventory.csv",
    index=False
)

In [20]:
candidates = projects_df[
    (projects_df["issues"] >= 1000)
    &
    (projects_df["resolved"] >= 500)
    &
    (projects_df["unresolved"] >= 50)
].copy()

In [21]:
len(candidates)

candidates.sort_values(
    "issues",
    ascending=False
).head(30)

,repository,project,issues,resolved,unresolved,resolved_rate,censoring_rate
844,Mojang,MC,213845,205547,8298,0.961196,0.038804
841,Mojang,MCPE,144778,140317,4461,0.969187,0.030813
884,Qt,QTBUG,97172,77520,19652,0.797761,0.202239
1192,Sonatype,OSSRH,74960,72193,2767,0.963087,0.036913
867,MongoDB,SERVER,58928,52972,5956,0.898928,0.101072
690,Jira,JRASERVER,47225,38678,8547,0.819015,0.180985
710,Jira,CONFSERVER,43910,37290,6620,0.849237,0.150763
1180,Sakai,SAK,43351,39086,4265,0.901617,0.098383
293,Apache,SPARK,37443,35034,2409,0.935662,0.064338
313,Apache,FLEX,35390,31564,3826,0.891890,0.108110
